## Vector Database setup

Remove old Weaviate DB files

In [16]:
!rm -rf ~/.local/share/weaviate

{"action":"read_disk_use","level":"error","msg":"failed to read disk usage: no such file or directory","path":"/home/jovyan/.local/share/weaviate","time":"2026-06-16T18:05:09Z"}



### Step 1 - Download sample data

In [17]:
import requests
import json

# Download the data
resp = requests.get('https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json')
data = json.loads(resp.text)  # Load data

# Parse the JSON and preview it
print(type(data), len(data))

def json_print(data):
    print(json.dumps(data, indent=2))

json_print(data[0])

<class 'list'> 10
{
  "Category": "SCIENCE",
  "Question": "This organ removes excess glucose from the blood & stores it as glycogen",
  "Answer": "Liver"
}


### Step 2 - Create an embedded instance of Weaviate vector database

In [18]:
import weaviate, os
from weaviate import EmbeddedOptions
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

client = weaviate.Client(
    embedded_options=EmbeddedOptions(),
    additional_headers={
        "X-OpenAI-BaseURL": os.environ['OPENAI_API_BASE'],
        "X-OpenAI-Api-Key": openai.api_key  # Replace this with your actual key
    }
)
print(f"Client created? {client.is_ready()}")

embedded weaviate is already listening on port 8079
Client created? True


In [19]:
json_print(client.get_meta())

{
  "hostname": "http://127.0.0.1:8079",
  "modules": {
    "generative-openai": {
      "documentationHref": "https://platform.openai.com/docs/api-reference/completions",
      "name": "Generative Search - OpenAI"
    },
    "qna-openai": {
      "documentationHref": "https://platform.openai.com/docs/api-reference/completions",
      "name": "OpenAI Question & Answering Module"
    },
    "ref2vec-centroid": {},
    "reranker-cohere": {
      "documentationHref": "https://txt.cohere.com/rerank/",
      "name": "Reranker - Cohere"
    },
    "text2vec-cohere": {
      "documentationHref": "https://docs.cohere.ai/embedding-wiki/",
      "name": "Cohere Module"
    },
    "text2vec-huggingface": {
      "documentationHref": "https://huggingface.co/docs/api-inference/detailed_parameters#feature-extraction-task",
      "name": "Hugging Face Module"
    },
    "text2vec-openai": {
      "documentationHref": "https://platform.openai.com/docs/guides/embeddings/what-are-embeddings",
      "nam

## Step 3 - Create Question collection

In [20]:
# resetting the schema. CAUTION: This will delete your collection 
if client.schema.exists("Question"):
    client.schema.delete_class("Question")
class_obj = {
    "class": "Question",
    "vectorizer": "text2vec-openai",  # Use OpenAI as the vectorizer
    "moduleConfig": {
        "text2vec-openai": {
            "model": "ada",
            "modelVersion": "002",
            "type": "text",
            "baseURL": os.environ["OPENAI_API_BASE"]
        }
    }
}

client.schema.create_class(class_obj)

time="2026-06-16T18:05:09Z" level=error msg="stop lsmkv store: shutdown bucket \"property_answer_searchable\": delete commit log file: remove /home/jovyan/.local/share/weaviate/question_JsvwmwcwyMTB_lsm/property_answer_searchable/segment-1781632815766564882.wal: no such file or directory" action=drop_shard class=Question id=question_JsvwmwcwyMTB
{"action":"hnsw_vector_cache_prefill","count":1000,"index_id":"question_PSQKNt6hfEj6","level":"info","limit":1000000000000,"msg":"prefilled vector cache","time":"2026-06-16T18:05:09Z","took":63021}


## Step 4 - Load sample data and generate vector embeddings

In [21]:
# reminder for the data structure
json_print(data[0])

{
  "Category": "SCIENCE",
  "Question": "This organ removes excess glucose from the blood & stores it as glycogen",
  "Answer": "Liver"
}


In [22]:
with client.batch.configure(batch_size=5) as batch:
    for i, d in enumerate(data):  # Batch import data
        
        print(f"importing question: {i+1}")
        
        properties = {
            "answer": d["Answer"],
            "question": d["Question"],
            "category": d["Category"],
        }
        
        batch.add_data_object(
            data_object=properties,
            class_name="Question"
        )

{"action":"restapi_management","level":"info","msg":"Shutting down... ","time":"2026-06-16T18:05:09Z"}
{"action":"restapi_management","level":"info","msg":"Stopped serving weaviate at http://127.0.0.1:8079","time":"2026-06-16T18:05:09Z"}
Exception in thread batchSizeRefresh:
Traceback (most recent call last):
  File "/usr/local/lib/python3.9/site-packages/urllib3/connection.py", line 174, in _new_conn
    conn = connection.create_connection(
  File "/usr/local/lib/python3.9/site-packages/urllib3/util/connection.py", line 95, in create_connection
    raise err
  File "/usr/local/lib/python3.9/site-packages/urllib3/util/connection.py", line 85, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py", line 715, in urlopen
    httplib_response = self._make_request(
  File "

importing question: 1
importing question: 2
importing question: 3
importing question: 4
importing question: 5
Embedded weaviate wasn't listening on port 8079, so starting embedded weaviate again
Started /home/jovyan/.cache/weaviate-embedded: process ID 237
importing question: 6
importing question: 7
importing question: 8
importing question: 9
importing question: 10


{"action":"startup","default_vectorizer_module":"none","level":"info","msg":"the default vectorizer modules is set to \"none\", as a result all new schema classes without an explicit vectorizer setting, will use this vectorizer","time":"2026-06-16T18:05:10Z"}
{"action":"startup","auto_schema_enabled":true,"level":"info","msg":"auto schema enabled setting is set to \"true\"","time":"2026-06-16T18:05:10Z"}
{"level":"warning","msg":"Multiple vector spaces are present, GraphQL Explore and REST API list objects endpoint module include params has been disabled as a result.","time":"2026-06-16T18:05:10Z"}
{"action":"grpc_startup","level":"info","msg":"grpc server listening at [::]:50060","time":"2026-06-16T18:05:10Z"}
{"action":"restapi_management","level":"info","msg":"Serving weaviate at http://127.0.0.1:8079","time":"2026-06-16T18:05:10Z"}
{"action":"hnsw_vector_cache_prefill","count":1000,"index_id":"question_cfoOgRacfHlI","level":"info","limit":1000000000000,"msg":"prefilled vector cache

In [23]:
count = client.query.aggregate("Question").with_meta_count().do()
json_print(count)

{
  "data": {
    "Aggregate": {
      "Question": [
        {
          "meta": {
            "count": 10
          }
        }
      ]
    }
  }
}


## Let's Extract the vector that represents each question!

In [24]:
# write a query to extract the vector for a question
result = (client.query
          .get("Question", ["category", "question", "answer"])
          .with_additional("vector")
          .with_limit(1)
          .do())

json_print(result)

{
  "data": {
    "Get": {
      "Question": [
        {
          "_additional": {
            "vector": []
          },
          "answer": "Sound barrier",
          "category": "SCIENCE",
          "question": "In 70-degree air, a plane traveling at about 1,130 feet per second breaks it"
        }
      ]
    }
  }
}


## Query time
What is the distance between the `query`: `biology` and the returned objects?

In [25]:
response = (
    client.query
    .get("Question",["question","answer","category"])
    .with_near_text({"concepts": "biology"})
    .with_additional('distance')
    .with_limit(2)
    .do()
)

json_print(response)

{
  "errors": [
    {
      "locations": [
        {
          "column": 24,
          "line": 1
        }
      ],
      "message": "Unknown argument \"nearText\" on field \"Question\" of type \"GetObjectsObj\". Did you mean \"nearVector\" or \"nearObject\"?",
      "path": null
    }
  ]
}


{"action":"requests_total","api":"graphql","class_name":"","error":"Unknown argument \"nearText\" on field \"Question\" of type \"GetObjectsObj\". Did you mean \"nearVector\" or \"nearObject\"?","level":"error","msg":"unexpected error","query_type":"","time":"2026-06-16T18:05:10Z"}


In [26]:
response = (
    client.query
    .get("Question", ["question", "answer"])
    .with_near_text({"concepts": ["animals"]})
    .with_limit(10)
    .with_additional(["distance"])
    .do()
)

json_print(response)

{
  "errors": [
    {
      "locations": [
        {
          "column": 25,
          "line": 1
        }
      ],
      "message": "Unknown argument \"nearText\" on field \"Question\" of type \"GetObjectsObj\". Did you mean \"nearVector\" or \"nearObject\"?",
      "path": null
    }
  ]
}


{"action":"requests_total","api":"graphql","class_name":"","error":"Unknown argument \"nearText\" on field \"Question\" of type \"GetObjectsObj\". Did you mean \"nearVector\" or \"nearObject\"?","level":"error","msg":"unexpected error","query_type":"","time":"2026-06-16T18:05:10Z"}


## We can let the vector database know to remove results after a threshold distance!

In [27]:
response = (
    client.query
    .get("Question", ["question", "answer"])
    .with_near_text({"concepts": ["animals"], "distance": 0.24})
    .with_limit(10)
    .with_additional(["distance"])
    .do()
)

json_print(response)

{
  "errors": [
    {
      "locations": [
        {
          "column": 25,
          "line": 1
        }
      ],
      "message": "Unknown argument \"nearText\" on field \"Question\" of type \"GetObjectsObj\". Did you mean \"nearVector\" or \"nearObject\"?",
      "path": null
    }
  ]
}


{"action":"requests_total","api":"graphql","class_name":"","error":"Unknown argument \"nearText\" on field \"Question\" of type \"GetObjectsObj\". Did you mean \"nearVector\" or \"nearObject\"?","level":"error","msg":"unexpected error","query_type":"","time":"2026-06-16T18:05:10Z"}


## Vector Databases support for CRUD operations

### Create

In [28]:
#Create an object
object_uuid = client.data_object.create(
    data_object={
        'question':"Leonardo da Vinci was born in this country.",
        'answer': "Italy",
        'category': "Culture"
    },
    class_name="Question"
 )

In [29]:
print(object_uuid)

d0f2c033-90d8-46f4-863f-262ed2ae113a


### Read

In [30]:
data_object = client.data_object.get_by_id(object_uuid, class_name="Question")
json_print(data_object)

{
  "class": "Question",
  "creationTimeUnix": 1781633110096,
  "id": "d0f2c033-90d8-46f4-863f-262ed2ae113a",
  "lastUpdateTimeUnix": 1781633110096,
  "properties": {
    "answer": "Italy",
    "category": "Culture",
    "question": "Leonardo da Vinci was born in this country."
  },
  "vectorWeights": null
}


In [31]:
data_object = client.data_object.get_by_id(
    object_uuid,
    class_name='Question',
    with_vector=True
)

json_print(data_object)

{
  "class": "Question",
  "creationTimeUnix": 1781633110096,
  "id": "d0f2c033-90d8-46f4-863f-262ed2ae113a",
  "lastUpdateTimeUnix": 1781633110096,
  "properties": {
    "answer": "Italy",
    "category": "Culture",
    "question": "Leonardo da Vinci was born in this country."
  },
  "vectorWeights": null
}


### Update

In [32]:
client.data_object.update(
    uuid=object_uuid,
    class_name="Question",
    data_object={
        'answer':"Florence, Italy"
    })

In [33]:
data_object = client.data_object.get_by_id(
    object_uuid,
    class_name='Question',
)

json_print(data_object)

{
  "class": "Question",
  "creationTimeUnix": 1781633110096,
  "id": "d0f2c033-90d8-46f4-863f-262ed2ae113a",
  "lastUpdateTimeUnix": 1781633110115,
  "properties": {
    "answer": "Florence, Italy",
    "category": "Culture",
    "question": "Leonardo da Vinci was born in this country."
  },
  "vectorWeights": null
}


### Delete

In [34]:
json_print(client.query.aggregate("Question").with_meta_count().do())

{
  "data": {
    "Aggregate": {
      "Question": [
        {
          "meta": {
            "count": 11
          }
        }
      ]
    }
  }
}


In [35]:
client.data_object.delete(uuid=object_uuid, class_name="Question")

In [36]:
json_print(client.query.aggregate("Question").with_meta_count().do())

{
  "data": {
    "Aggregate": {
      "Question": [
        {
          "meta": {
            "count": 10
          }
        }
      ]
    }
  }
}
